In [1]:
import pandas as pd
import numpy as np


In [2]:
# === 1) LOAD ERA5 DAILY DATA ===
# Change the file name here if yours is different:
path = "data/ERA5_daily/era5_daily_clean.csv"

df = pd.read_csv(path)

# Check columns
print(df.columns)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'data/ERA5_daily/era5_daily_clean.csv'

In [3]:
import pandas as pd
import numpy as np

# === 1) LOAD ERA5 DAILY DATA (EXCEL VERSION) ===
# Adjusted path based on your folder structure:
path = "data/ERA5_clean/ERA5_Milan_1990_2024_daily.xlsx"

df = pd.read_excel(path)   # if it complains about engine, use engine="openpyxl"

print(df.columns)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'data/ERA5_clean/ERA5_Milan_1990_2024_daily.xlsx'

In [4]:
import pandas as pd
import numpy as np

# === 1) LOAD ERA5 DAILY DATA (EXCEL) ===
path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\ERA5_clean\ERA5_Milan_1990_2024_daily.xlsx"

df = pd.read_excel(path)   # if you get an engine warning, add: engine="openpyxl"

print(df.columns)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'F:\\Building Engineering - Polytechnic University of Turin\\Masters Thesis\\23-11-2025\\PythonProjects\\ClimateProcessingProject\\data\\ERA5_clean\\ERA5_Milan_1990_2024_daily.xlsx'

In [5]:
import os

folder = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\ERA5_clean"
print(os.listdir(folder))


['ERA5_Milan_1990_2024_daily.csv', 'TEXT 1.docx', 'TEXT 10.docx', 'TEXT 11.docx', 'TEXT 12.docx', 'TEXT 13.docx', 'TEXT 14.docx', 'TEXT 2.docx', 'TEXT 3.docx', 'TEXT 4.docx', 'TEXT 5.docx', 'TEXT 6.docx', 'TEXT 7.docx', 'TEXT 8.docx', 'TEXT 9.docx']


In [6]:
import pandas as pd
import numpy as np

# === 1) LOAD ERA5 DAILY DATA (CSV) ===
path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\ERA5_clean\ERA5_Milan_1990_2024_daily.csv"

df = pd.read_csv(path)

print(df.columns)
df.head()


Index(['time', 'dewpoint_temperature_2m', 'evaporation_from_bare_soil',
       'evaporation_from_open_water_surfaces_excluding_oceans',
       'potential_evaporation', 'skin_temperature',
       'surface_net_solar_radiation', 'surface_net_thermal_radiation',
       'surface_pressure', 'surface_solar_radiation_downwards',
       'surface_thermal_radiation_downwards', 'temperature_2m',
       'total_evaporation', 'total_precipitation', 'u_component_of_wind_10m',
       'v_component_of_wind_10m', 'volumetric_soil_water_layer_1',
       'wind_speed_10m', 'year'],
      dtype='object')


,time,dewpoint_temperature_2m,evaporation_from_bare_soil,evaporation_from_open_water_surfaces_excluding_oceans,potential_evaporation,skin_temperature,surface_net_solar_radiation,surface_net_thermal_radiation,surface_pressure,surface_solar_radiation_downwards,surface_thermal_radiation_downwards,temperature_2m,total_evaporation,total_precipitation,u_component_of_wind_10m,v_component_of_wind_10m,volumetric_soil_water_layer_1,wind_speed_10m,year
0,1990-01-01,-5.056007,-0.000002,-0.000013,-0.000309,-2.650599,693.907724,-683.822209,100014.817097,802.644514,3076.099459,-3.189028,-0.000021,0.003621,0.998954,-0.258948,0.430315,1.145791,1990
1,1990-01-02,-4.943282,-0.000002,-0.000011,-0.000315,-2.590518,776.261460,-780.820247,99882.806251,897.531968,2978.141673,-2.539750,-0.000021,0.002283,0.786257,-0.345120,0.429229,1.023357,1990
2,1990-01-03,-4.455809,-0.000003,-0.000011,-0.000303,-1.993348,822.563560,-808.627321,99968.854107,950.753098,2977.127017,-1.654876,-0.000019,0.000487,-0.111026,-0.500728,0.428195,1.012839,1990
3,1990-01-04,-2.678010,-0.000004,-0.000012,-0.000240,-1.117184,636.098449,-529.449444,100598.551367,734.760057,3332.996628,-0.925320,-0.000022,0.000744,0.562771,-0.712550,0.427213,1.293427,1990
4,1990-01-05,-4.062554,-0.000004,-0.000013,-0.000278,-2.345535,736.553352,-694.557060,100724.552069,850.630783,3098.511624,-2.218500,-0.000025,0.003279,0.127955,-0.194423,0.426276,0.994992,1990


In [7]:
# Convert time to datetime
df["time"] = pd.to_datetime(df["time"])

# Convert Kelvin → °C if needed
def to_celsius(series):
    if series.mean() > 200:  # probably Kelvin
        return series - 273.15
    return series

df["t_mean"] = to_celsius(df["temperature_2m"])
df["t_dew"] = to_celsius(df["dewpoint_temperature_2m"])

df["date"] = df["time"]
df["month"] = df["date"].dt.month
df["year"] = df["year"].astype(int)  # already present, but ensure integer


In [8]:
import numpy as np

def compute_rh(t, t_dew):
    a, b = 17.625, 243.04
    gamma_t = (a * t) / (b + t)
    gamma_td = (a * t_dew) / (b + t_dew)
    rh = 100 * np.exp(gamma_td - gamma_t)
    return np.clip(rh, 0, 100)

df["RH"] = compute_rh(df["t_mean"], df["t_dew"])


In [9]:
def compute_heat_index(t_c, rh):
    T = t_c * 9/5 + 32  # C→F
    HI_F = (-42.379 + 2.04901523*T + 10.14333127*rh
            - 0.22475541*T*rh - 6.83783e-3*T*T
            - 5.481717e-2*rh*rh + 1.22874e-3*T*T*rh
            + 8.5282e-4*T*rh*rh - 1.99e-6*T*T*rh*rh)
    HI_F = np.where(T < 80, T, HI_F)
    return (HI_F - 32) * 5/9  # back to °C

df["HI"] = compute_heat_index(df["t_mean"], df["RH"])


In [10]:
df["CDD26"] = np.where(df["t_mean"] > 26, df["t_mean"] - 26, 0)
df["HDD15"] = np.where(df["t_mean"] < 15, 15 - df["t_mean"], 0)
df["HDD18"] = np.where(df["t_mean"] < 18, 18 - df["t_mean"], 0)


In [11]:
annual = df.groupby("year").agg(
    mean_temp=("t_mean", "mean"),
    max_temp=("t_mean", "max"),
    mean_temp_summer=("t_mean", lambda x: x[df.loc[x.index, "month"].isin([6,7,8])].mean()),
    CDD26=("CDD26", "sum"),
    HDD15=("HDD15", "sum"),
    HDD18=("HDD18", "sum"),
    RH_mean=("RH", "mean"),
    HI_mean=("HI", "mean")
).reset_index()

annual.head()


,year,mean_temp,max_temp,mean_temp_summer,CDD26,HDD15,HDD18,RH_mean,HI_mean
0,1990,12.495180,26.392913,21.592681,1.116915,1719.190283,2391.860321,73.616339,12.495180
1,1991,12.335233,28.712203,23.552264,31.984059,1956.936792,2682.196208,70.108965,12.360321
2,1992,12.438178,27.000560,21.517723,3.206055,1735.089324,2429.239361,75.743024,12.443362
3,1993,12.296997,27.567648,21.989618,9.351023,1778.929793,2479.281325,73.164731,12.303090
4,1994,13.307641,28.747541,23.427990,20.415498,1545.500970,2257.008500,74.488525,13.331643


In [12]:
annual.to_csv(
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\ERA5_clean\climate_features_annual.csv",
    index=False
)


In [13]:
print(df.columns)


Index(['time', 'dewpoint_temperature_2m', 'evaporation_from_bare_soil',
       'evaporation_from_open_water_surfaces_excluding_oceans',
       'potential_evaporation', 'skin_temperature',
       'surface_net_solar_radiation', 'surface_net_thermal_radiation',
       'surface_pressure', 'surface_solar_radiation_downwards',
       'surface_thermal_radiation_downwards', 'temperature_2m',
       'total_evaporation', 'total_precipitation', 'u_component_of_wind_10m',
       'v_component_of_wind_10m', 'volumetric_soil_water_layer_1',
       'wind_speed_10m', 'year', 't_mean', 't_dew', 'date', 'month', 'RH',
       'HI', 'CDD26', 'HDD15', 'HDD18'],
      dtype='object')


In [14]:
# === 8) SKIN TEMPERATURE (Ts) IN °C ===
df["ts"] = to_celsius(df["skin_temperature"])

# Annual and summer means of Ts
ts_annual = df.groupby("year").agg(
    ts_mean=("ts", "mean"),
    ts_summer=("ts", lambda x: x[df.loc[x.index, "month"].isin([6, 7, 8])].mean())
).reset_index()

ts_annual.head()


,year,ts_mean,ts_summer
0,1990,12.340825,22.013332
1,1991,12.189402,24.144450
2,1992,12.283295,21.879677
3,1993,11.978588,22.164053
4,1994,13.172501,23.927467


In [15]:
# === 9) BASELINE (CLIMATOLOGY) FOR SKIN TEMPERATURE ===
baseline_mask = (ts_annual["year"] >= 1991) & (ts_annual["year"] <= 2020)

ts_baseline_mean = ts_annual.loc[baseline_mask, "ts_mean"].mean()
ts_baseline_summer_mean = ts_annual.loc[baseline_mask, "ts_summer"].mean()

print("Baseline Ts annual mean (1991–2020):", ts_baseline_mean)
print("Baseline Ts summer mean (1991–2020):", ts_baseline_summer_mean)

# === 10) ANOMALIES ===
ts_annual["ts_anom"] = ts_annual["ts_mean"] - ts_baseline_mean
ts_annual["ts_anom_summer"] = ts_annual["ts_summer"] - ts_baseline_summer_mean

ts_annual.head()


Baseline Ts annual mean (1991–2020): 12.711380277968404
Baseline Ts summer mean (1991–2020): 23.139559014149494


,year,ts_mean,ts_summer,ts_anom,ts_anom_summer
0,1990,12.340825,22.013332,-0.370555,-1.126227
1,1991,12.189402,24.144450,-0.521978,1.004891
2,1992,12.283295,21.879677,-0.428085,-1.259882
3,1993,11.978588,22.164053,-0.732792,-0.975506
4,1994,13.172501,23.927467,0.461121,0.787908


In [16]:
# === 11) MERGE Ts ANOMALIES INTO CLIMATE FEATURES ===
annual = annual.merge(
    ts_annual[["year", "ts_anom", "ts_anom_summer"]],
    on="year",
    how="left"
)

annual.head()


,year,mean_temp,max_temp,mean_temp_summer,CDD26,HDD15,HDD18,RH_mean,HI_mean,ts_anom,ts_anom_summer
0,1990,12.495180,26.392913,21.592681,1.116915,1719.190283,2391.860321,73.616339,12.495180,-0.370555,-1.126227
1,1991,12.335233,28.712203,23.552264,31.984059,1956.936792,2682.196208,70.108965,12.360321,-0.521978,1.004891
2,1992,12.438178,27.000560,21.517723,3.206055,1735.089324,2429.239361,75.743024,12.443362,-0.428085,-1.259882
3,1993,12.296997,27.567648,21.989618,9.351023,1778.929793,2479.281325,73.164731,12.303090,-0.732792,-0.975506
4,1994,13.307641,28.747541,23.427990,20.415498,1545.500970,2257.008500,74.488525,13.331643,0.461121,0.787908


In [17]:
# === 12) SAVE UPDATED CLIMATE FEATURES (WITH Ts ANOM) ===
output_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\ERA5_clean\climate_features_annual.csv"
annual.to_csv(output_path, index=False)
output_path


'F:\\Building Engineering - Polytechnic University of Turin\\Masters Thesis\\23-11-2025\\PythonProjects\\ClimateProcessingProject\\data\\ERA5_clean\\climate_features_annual.csv'

In [1]:
import pandas as pd

climate_annual_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\ERA5_clean\climate_features_annual.csv"

clim = pd.read_csv(climate_annual_path)

clim.head(), clim["year"].min(), clim["year"].max()


(   year  mean_temp   max_temp  mean_temp_summer      CDD26        HDD15  \
 0  1990  12.495180  26.392913         21.592681   1.116915  1719.190283   
 1  1991  12.335233  28.712203         23.552264  31.984059  1956.936792   
 2  1992  12.438178  27.000560         21.517723   3.206055  1735.089324   
 3  1993  12.296997  27.567648         21.989618   9.351023  1778.929793   
 4  1994  13.307641  28.747541         23.427990  20.415498  1545.500970   
 
          HDD18    RH_mean    HI_mean   ts_anom  ts_anom_summer  
 0  2391.860321  73.616339  12.495180 -0.370555       -1.126227  
 1  2682.196208  70.108965  12.360321 -0.521978        1.004891  
 2  2429.239361  75.743024  12.443362 -0.428085       -1.259882  
 3  2479.281325  73.164731  12.303090 -0.732792       -0.975506  
 4  2257.008500  74.488525  13.331643  0.461121        0.787908  ,
 np.int64(1990),
 np.int64(2024))

In [2]:
# Select the climatology period 2005–2020
mask = (clim["year"] >= 2005) & (clim["year"] <= 2020)
clim_period = clim.loc[mask].copy()

clim_period[["year"] + [c for c in clim_period.columns if c != "year"]].head()


,year,mean_temp,max_temp,mean_temp_summer,CDD26,HDD15,HDD18,RH_mean,HI_mean,ts_anom,ts_anom_summer
15,2005,12.395614,28.951722,22.870312,19.527296,1895.430068,2572.665748,70.360034,12.418626,-0.930323,-0.111389
16,2006,13.134494,30.085083,23.408803,48.233776,1696.923352,2367.985861,70.694037,13.146286,0.250617,0.997743
17,2007,13.466953,28.724500,22.269120,17.767430,1505.304151,2142.467373,69.625337,13.458512,0.340683,-0.668762
18,2008,12.737211,25.836098,21.833431,0.000000,1603.688379,2339.938601,75.546570,12.737211,-0.232346,-0.952692
19,2009,13.246819,28.452726,23.445496,11.753503,1724.330291,2355.792678,70.910098,13.261191,0.184925,0.756734


In [3]:
# Compute multi-year mean of all indicators (except 'year')
clim_clim = clim_period.drop(columns=["year"]).mean().to_frame().T

# Add a period label
clim_clim.insert(0, "period", "2005-2020")

clim_clim


,period,mean_temp,max_temp,mean_temp_summer,CDD26,HDD15,HDD18,RH_mean,HI_mean,ts_anom,ts_anom_summer
0,2005-2020,13.214608,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028


In [5]:
clim_clim_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\ERA5_clean\climate_features_climatology_2005_2020.csv"

clim_clim.to_csv(clim_clim_path, index=False)
clim_clim_path


'F:\\Building Engineering - Polytechnic University of Turin\\Masters Thesis\\23-11-2025\\PythonProjects\\ClimateProcessingProject\\data\\ERA5_clean\\climate_features_climatology_2005_2020.csv'

In [6]:
# ============================================
# STAGE 10.6 — ATTACH CLIMATE TO BUILDINGS
# 10.6.1 — Load buildings + climate climatology
# ============================================

import geopandas as gpd
import pandas as pd

# Paths
buildings_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_geom_features.geojson"
clim_clim_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\ERA5_clean\climate_features_climatology_2005_2020.csv"

# Load datasets
buildings = gpd.read_file(buildings_path)
clim_clim = pd.read_csv(clim_clim_path)

# Display heads for confirmation
buildings.head(), clim_clim



(  STRATO TEMA CLASSE       ID_ZRIL    FEATURE_ID building_type building_use  \
 0     02   01     02  MI_2006_0033  000000000001          0101         0291   
 1     02   01     02  MI_2006_0033  000000000002          0101         0201   
 2     02   01     02  MI_2006_0033  000000000003          0101         0201   
 3     02   01     02  MI_2006_0033  000000000004          0101         0201   
 4     02   01     02  MI_2006_0033  000000000005          0101         0201   
 
   building_status   building_id EDIFC_CASS     area_m2  centroid_lon  \
 0            0401  E00000000001  000000001  727.719472      9.223222   
 1            0403  E00000000002  000000002  270.704927      9.234857   
 2            0403  E00000000003  000000003  281.929276      9.220783   
 3            0403  E00000000004  000000004  347.607947      9.220984   
 4            0403  E00000000005  000000005  191.138372      9.220669   
 
    centroid_lat                                           geometry  
 0     4

In [7]:
# ============================================
# 10.6.2 — Broadcast climate indicators to buildings
# ============================================

# Take the single climatology row as a Series
clim_row = clim_clim.iloc[0]

# Add each climate feature as a new column to the buildings dataframe
for col in clim_clim.columns:
    if col == "period":
        continue  # keep 'period' out as metadata if desired
    buildings[col] = float(clim_row[col])

# Check the output: buildings now have climate columns
buildings[[
    "building_id", "area_m2", 
    "mean_temp", "max_temp", "mean_temp_summer",
    "CDD26", "HDD15", "HDD18",
    "RH_mean", "HI_mean",
    "ts_anom", "ts_anom_summer"
]].head()


,building_id,area_m2,mean_temp,max_temp,mean_temp_summer,CDD26,HDD15,HDD18,RH_mean,HI_mean,ts_anom,ts_anom_summer
0,E00000000001,727.719472,13.214608,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028
1,E00000000002,270.704927,13.214608,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028
2,E00000000003,281.929276,13.214608,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028
3,E00000000004,347.607947,13.214608,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028
4,E00000000005,191.138372,13.214608,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028


In [8]:
# ============================================
# 10.6.3 — Save Buildings + Climate dataset
# ============================================

buildings_out_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate.geojson"

buildings.to_file(buildings_out_path, driver="GeoJSON")

print("Saved integrated dataset to:")
print(buildings_out_path)


Saved integrated dataset to:
F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate.geojson


In [9]:
# OPTIONAL: overwrite and re-export buildings_with_climate.geojson
buildings.to_file("F:/Building Engineering - Polytechnic University of Turin/Masters Thesis/23-11-2025/PythonProjects/ClimateProcessingProject/data/Buildings/buildings_with_climate.geojson", driver="GeoJSON")


In [10]:
import geopandas as gpd
from shapely.geometry import shape, mapping
import json

# Load raw GeoJSON
with open("buildings_with_climate.geojson", "r", encoding="utf-8") as f:
    raw = json.load(f)

# Flatten 3D geometries
for feat in raw["features"]:
    geom = shape(feat["geometry"])
    feat["geometry"] = mapping(geom.from_bounds(*geom.bounds))  # safe simplification

# Re-save as clean 2D
with open("buildings_with_climate_2d.geojson", "w", encoding="utf-8") as f:
    json.dump(raw, f)


FileNotFoundError: [Errno 2] No such file or directory: 'buildings_with_climate.geojson'

In [11]:
import geopandas as gpd
from shapely.geometry import shape, mapping
import json

# FULL PATH to your original file (update this to your exact path)
input_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate.geojson"
output_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate_2d.geojson"

# Load raw GeoJSON
with open(input_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Drop Z-values from geometry
def flatten_coords(geom):
    if geom["type"] == "Polygon":
        return {
            "type": "Polygon",
            "coordinates": [
                [(x, y) for x, y, *_ in ring]
                for ring in geom["coordinates"]
            ]
        }
    elif geom["type"] == "MultiPolygon":
        return {
            "type": "MultiPolygon",
            "coordinates": [
                [
                    [(x, y) for x, y, *_ in ring]
                    for ring in poly
                ]
                for poly in geom["coordinates"]
            ]
        }
    return geom

# Apply cleaning
for feature in raw["features"]:
    feature["geometry"] = flatten_coords(feature["geometry"])

# Save cleaned version
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(raw, f)

print("✅ Clean 2D version saved as:", output_path)


✅ Clean 2D version saved as: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate_2d.geojson


In [12]:
import geopandas as gpd

buildings_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate_2d.geojson"
buildings_gdf = gpd.read_file(buildings_path)

buildings_gdf.info()
buildings_gdf.head()


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 53041 entries, 0 to 53040
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   STRATO            53041 non-null  object  
 1   TEMA              53041 non-null  object  
 2   CLASSE            53041 non-null  object  
 3   ID_ZRIL           53041 non-null  object  
 4   FEATURE_ID        53041 non-null  object  
 5   building_type     53041 non-null  object  
 6   building_use      53041 non-null  object  
 7   building_status   53041 non-null  object  
 8   building_id       53041 non-null  object  
 9   EDIFC_CASS        53041 non-null  object  
 10  area_m2           53041 non-null  float64 
 11  centroid_lon      53041 non-null  float64 
 12  centroid_lat      53041 non-null  float64 
 13  mean_temp         53041 non-null  float64 
 14  max_temp          53041 non-null  float64 
 15  mean_temp_summer  53041 non-null  float64 
 16  CDD26         

,STRATO,TEMA,CLASSE,ID_ZRIL,FEATURE_ID,building_type,building_use,building_status,building_id,EDIFC_CASS,...,max_temp,mean_temp_summer,CDD26,HDD15,HDD18,RH_mean,HI_mean,ts_anom,ts_anom_summer,geometry
0,02,01,02,MI_2006_0033,000000000001,0101,0291,0401,E00000000001,000000001,...,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028,"POLYGON ((9.22342 45.49639, 9.22342 45.49639, ..."
1,02,01,02,MI_2006_0033,000000000002,0101,0201,0403,E00000000002,000000002,...,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028,"POLYGON ((9.23502 45.49384, 9.23492 45.49376, ..."
2,02,01,02,MI_2006_0033,000000000003,0101,0201,0403,E00000000003,000000003,...,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028,"POLYGON ((9.22071 45.49534, 9.22097 45.49528, ..."
3,02,01,02,MI_2006_0033,000000000004,0101,0201,0403,E00000000004,000000004,...,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028,"POLYGON ((9.22097 45.49546, 9.22097 45.49546, ..."
4,02,01,02,MI_2006_0033,000000000005,0101,0201,0403,E00000000005,000000005,...,28.806782,22.961018,21.252445,1607.729338,2280.836131,72.844047,13.241382,0.157789,0.109028,"POLYGON ((9.22076 45.49551, 9.22076 45.4955, 9..."


In [13]:
# Load clean buildings + climate GeoJSON (already validated locally)
buildings_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate_2d.geojson"
buildings_gdf = gpd.read_file(buildings_path)

# Normalize building address column
buildings_gdf["INDIRIZZO_NORM"] = buildings_gdf["EDIFC_CASS"].astype(str).str.strip().str.upper()


In [14]:
# Save cleaned building+climate file for EPC merge
buildings_gdf.to_file(
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_ready_for_epc.geojson",
    driver="GeoJSON"
)


In [15]:
# Save the buildings GeoDataFrame as a clean CSV with centroid info only (for EPC join)
buildings_gdf[["building_id", "centroid_lon", "centroid_lat", "INDIRIZZO_NORM"]].to_csv(
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_address_reference.csv",
    index=False
)


In [16]:
# Save building reference info without geometry
buildings_gdf[["building_id", "centroid_lon", "centroid_lat", "INDIRIZZO_NORM"]].to_csv(
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_address_reference.csv",
    index=False
)


In [17]:
buildings_gdf[["EDIFC_CASS", "INDIRIZZO_NORM"]].head()


,EDIFC_CASS,INDIRIZZO_NORM
0,000000001,000000001
1,000000002,000000002
2,000000003,000000003
3,000000004,000000004
4,000000005,000000005


In [18]:
buildings_gdf[["building_id", "centroid_lon", "centroid_lat", "INDIRIZZO_NORM"]].to_csv(
    r"F:\...path...\buildings_address_reference.csv",
    index=False
)


OSError: Cannot save file into a non-existent directory: 'F:\...path...'

In [19]:
# Save clean building address reference with full real path
buildings_gdf[["building_id", "centroid_lon", "centroid_lat", "INDIRIZZO_NORM"]].to_csv(
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_address_reference.csv",
    index=False
)


PermissionError: [Errno 13] Permission denied: 'F:\\Building Engineering - Polytechnic University of Turin\\Masters Thesis\\23-11-2025\\PythonProjects\\ClimateProcessingProject\\data\\Buildings\\buildings_address_reference.csv'

In [20]:
# Save clean building address reference with full real path
buildings_gdf[["building_id", "centroid_lon", "centroid_lat", "INDIRIZZO_NORM"]].to_csv(
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_address_reference.csv",
    index=False
)


In [21]:
import pandas as pd

# File paths
file_a = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\epc_clean_final.csv"
file_b = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"

# Load first 1000 rows from each to keep it fast
epc_a = pd.read_csv(file_a, nrows=1000, low_memory=False)
epc_b = pd.read_csv(file_b, nrows=1000, low_memory=False)

# Print shape and sample columns
print("File A - epc_clean_final.csv")
print(f"Shape: {epc_a.shape}")
print("Columns:", list(epc_a.columns[:25]), "\n")

print("File B - Database_CENED+2...csv")
print(f"Shape: {epc_b.shape}")
print("Columns:", list(epc_b.columns[:25]), "\n")

# Check for potential spatial / ID matching columns
spatial_cols = ['Y', 'Z', 'lat', 'lon', 'wgs84_x', 'wgs84_y', 'foglio', 'particella', 'subalterno']

def find_relevant_cols(df, label):
    found = [col for col in df.columns if any(key.lower() in col.lower() for key in spatial_cols)]
    print(f"{label} — spatial/join-related columns:\n{found}\n")

find_relevant_cols(epc_a, "File A")
find_relevant_cols(epc_b, "File B")


File A - epc_clean_final.csv
Shape: (1000, 13)
Columns: ['COD_APE', 'DATA_INS', 'COMUNE', 'INDIRIZZO', 'INDIRIZZO_NORM', 'ANNO_COSTRUZIONE_CLEAN', 'EP_GL_NREN', 'EP_GL_REN', 'CLASSE_ENERGETICA', 'CLIMATIZZAZIONE_ESTIVA', 'SUPERF_UTILE_RAFFRESCATA', 'VOLUME_LORDO_RAFFRESCATO', 'CONSUMI_TELERAFFRESCAMENTO'] 

File B - Database_CENED+2...csv
Shape: (1000, 198)
Columns: ['COD_APE', 'DATA_INS', 'RESIDENZIALE', 'NON_RESIDENZIALE', 'COMUNE_CATASTALE', 'PROPRIETA_PUBBLICA', 'USO_PUBBLICO', 'CLASSIFICAZIONE_DPR', 'INTERO_EDIFICIO', 'UNITA_IMMOBILIARE', 'GRUPPO_UNITA_IMMOBILIARI', 'NUMERO_UNITA_IMMOBILIARI', 'NUOVA_COSTRUZIONE', 'PASSAGGIO_PROPRIETA', 'LOCAZIONE', 'RISTRUTTURAZIONE_IMPORTANTE', 'RIQUALIFICAZIONE_ENERGETICA', 'OGGETTO_ALTRO', 'OGGETTO_ALTRO_TXT', 'REGIONE', 'COMUNE', 'INDIRIZZO', 'PIANO', 'INTERNO', 'ZONA_CLIMATICA'] 

File A — spatial/join-related columns:
['INDIRIZZO', 'INDIRIZZO_NORM', 'ANNO_COSTRUZIONE_CLEAN', 'CLIMATIZZAZIONE_ESTIVA']

File B — spatial/join-related columns:
